# AXI test 26: Rajnish Choubey's witness quorum on Gemini 3.8 Flash
**What this is.** Concept-map item 122 built a witness quorum from Rajnish Choubey's article "The 99 Eyewitnesses of the Ramayana": several witnesses propose the change for one request, and a change is committed only when enough of them proposed exactly the same change. This notebook puts it on a real model. The job is Test 23's conversation in plain sentences (the same 400 steps and chatter), with Gemini's thinking set to **minimal** so that one reply slips more often. Two arms: **S**, one witness (AXI as in Test 23), and **Q**, witnesses (ask 2; if they propose the same change, commit it; if not, call more, up to 5). Test 23's saved runs are the reference and cost nothing.

The predictions and the reading rules were written before any call in `docs/GEMINI_QUORUM_TEST.md` (hash-locked).

**This test costs real money, and this notebook never turns billing on.** Steps: (1) type the prices in Cell 5; (2) run the pilot in Cell 6, which prints what the whole test would cost **at your prices**; (3) only then type a budget in dollars (`BUDGET_USD`, for example 5) and run Cells 7 and 8. The notebook refuses the next call once the saved answers cost the budget, and a second guard stops each run at a cap on tokens. Everything finished is saved and the same cell carries on.

Run each cell **one at a time, top to bottom**. **Keep this tab open** while a run is going. If it stops for any reason, open the notebook again, run Cells 2 to 5, then run the cell you were on again. No GPU is needed.

### Cell 1: put your key into Colab Secrets (once; no code to run)
Colab left bar -> key icon (*Secrets*) -> **Add new secret**. Name: `GOOGLE_API_KEY` (exactly). Value: your key from `aistudio.google.com` -> *API Keys*. Switch **Notebook access** on. If you did this for an earlier notebook it is already done. Never paste the key into a cell or into the chat.

### Cell 2: get the code from GitHub

In [ ]:
!rm -rf /content/axi && git clone -q --depth 1 https://github.com/gaurshalendra2412/axi-runtime /content/axi
import os
assert os.path.exists("/content/axi/python/axi/experiments/gemini_quorum.py"), "the code is not on GitHub yet: update 83 must be pushed first"
print("code ok")

### Cell 3: install the Gemini library (about 20 seconds)

In [ ]:
!pip -q install -U google-genai

### Cell 4: connect Google Drive (a pop-up asks permission: Connect / Allow). Your progress and every reply are kept there

In [ ]:
import sys
sys.path.insert(0, "/content/axi/python")
from colab.colab_persist import open_store
store = open_store("/content/out", "axi_gemini_quorum")

### Cell 5: settings. Fill the lines at the top. PAID is True only after you turned billing on. The budget stays empty until you have read the pilot

In [ ]:
import json, time
from google.colab import userdata, files
from axi.experiments import gemini_quorum as Q
from axi.experiments import gemini_fuzzy as F
from axi.experiments.api_llm import QuotaStop, ServiceBusy, BudgetStop

PAID = None          # True ONLY after you turned billing on for this key in Google AI Studio
PRICE_IN = None      # dollars per 1,000,000 INPUT tokens of Gemini 3.8 Flash (the AI Studio price page)
PRICE_OUT = None     # dollars per 1,000,000 OUTPUT tokens (thinking tokens are billed as output)
PRICE_CACHED = None  # dollars per 1,000,000 CACHED input tokens (optional here: this test sends a short prompt every step)
BUDGET_USD = None    # the most you allow this test to cost. Leave None until you have read the pilot in Cell 6
THINKING = Q.THINKING  # 'minimal'. Change to Q.THINKING_FALLBACK ('low') ONLY if the pilot stops with an error about thinking

assert PAID in (True, False), "Set PAID: True if you turned billing on, False for a free key."
try: KEY = userdata.get("GOOGLE_API_KEY")
except Exception: raise SystemExit("No key found. Do Cell 1: a Secret named GOOGLE_API_KEY with Notebook access on.")
if PAID: assert PRICE_IN and PRICE_OUT, "Type PRICE_IN and PRICE_OUT (dollars per million tokens). The notebook has no built-in price on purpose."
TAG = str(THINKING); CACHE = "axi_gemini_quorum_cache_%s.jsonl" % TAG
print("reply file:", store.restore(CACHE))
guard = dict(budget_usd=BUDGET_USD, price_in=PRICE_IN, price_out=PRICE_OUT, price_cached=PRICE_CACHED) if (PAID and BUDGET_USD) else dict(price_in=PRICE_IN, price_out=PRICE_OUT, price_cached=PRICE_CACHED)
llm, MODEL = F.GL.build_chat_llm(KEY, store.path(CACHE), daily_limit=None if PAID else 20, min_interval=0.3 if PAID else 4.5, on_write=lambda: store.persist(CACHE), thinking=THINKING, **guard)
META = dict(model=MODEL, thinking=THINKING)
sp = llm.spent_usd()
print("paid:", PAID, "| model:", MODEL, "| thinking:", THINKING, "| budget:", ("$%.2f" % BUDGET_USD) if BUDGET_USD else "NOT SET (pilot only)", "| spent so far with these settings:", ("$%.2f" % sp) if sp is not None else "-")
BUSY = "This is Google's server being busy. Wait at least 30 minutes, then run THIS SAME CELL once. Do not run it again and again."
def go(fn):
    try: return fn()
    except QuotaStop as e:
        print("\nSTOPPED FOR NOW: " + str(e))
        if isinstance(e, ServiceBusy): print(BUSY)
        return None
def need_budget(): assert PAID and BUDGET_USD, "Read the pilot (Cell 6), then type BUDGET_USD in Cell 5, run Cell 5 again, and run this cell again."
def save(res, name):
    p = store.path(name + ".json"); json.dump(res, open(p, "w")); store.persist(name + ".json"); return p
def persisting(name, inner):
    print("progress file:", store.restore(name + ".jsonl"))
    def cb(*a):
        inner(*a)
        if a[0] % 25 == 0 or a[0] == a[1]: store.persist(name + ".jsonl")
    return cb
def prog(t, n, r):
    if t % 20 == 0 or t == n:
        sp = llm.spent_usd(); print(f"step {t}/{n} | witnesses {r.get('heard', 1 + len(r.get('reasks', [])))} | {r['seconds']:.1f} s | held list exact {r['exact']} (distance {r['distance']}) | spent {('$%.2f' % sp) if sp is not None else '-'}")

### Cell 6: the PILOT. 20 live steps of each arm (cents). The first 60 steps are answered by the program so the list has its full size. It prints the thinking tokens a call (they must be small: the minimal thinking took effect), the witnesses a step, the seconds, and the projected cost of the whole test at your prices

In [ ]:
assert PAID, "The pilot needs a paid key (about 60 calls)."
ps = go(lambda: F.run_axi(llm, Q.SEED_PILOT, 80, store.path("q26_pilot_s.jsonl"), meta=META, scripted=60))
pq = go(lambda: Q.run_quorum(llm, Q.SEED_PILOT, 80, store.path("q26_pilot_q.jsonl"), meta=META, scripted=60))
if ps and pq:
    store.persist("q26_pilot_s.jsonl"); store.persist("q26_pilot_q.jsonl")
    F.print_stats("pilot S (one witness)", ps["rows"][60:]); F.print_stats("pilot Q (witnesses)", pq["rows"][60:]); Q.print_q(pq["rows"])
    rep = Q.pilot_report(ps["rows"], pq["rows"], PRICE_IN, PRICE_OUT)
    print()
    for k, v in rep.items(): print(f"  {k}: {v if not isinstance(v, float) else round(v, 3)}")
    if rep["s_think_per_call"] is not None and rep["s_think_per_call"] > Q.REF_THINK_PER_CALL / 4: print("\nNOTE: the thinking tokens a call are not small. Send me this output before running Cells 7 and 8.")
    print("\nThen type BUDGET_USD in Cell 5 (a little above cost_both, for example 5), run Cell 5 again, and run Cell 7.")

### Cell 7: ARM S, one witness. 400 steps, the exact list shown at every step, up to 2 re-asks. A line every 20 steps

In [ ]:
need_budget()
nameS = "q26_S_seed%d_%s" % (Q.SEED, TAG)
resS = go(lambda: F.run_axi(llm, Q.SEED, Q.STEPS, store.path(nameS + ".jsonl"), meta=META, max_tokens=Q.S_TOKEN_CAP, progress=persisting(nameS, prog)))
if resS:
    F.print_stats("ARM S (one witness)", resS["rows"]); F.print_kinds(resS["rows"]); store.persist(nameS + ".jsonl")
    print("legal-but-wrong commits:", Q.wrong_commits(resS["rows"]), "| spent so far on this model: $%.2f" % llm.spent_usd()); files.download(save(resS, nameS))

### Cell 8: ARM Q, the witnesses. 400 steps; 2 witnesses a step when they agree, up to 5 when they do not. A line every 20 steps

In [ ]:
need_budget()
nameQ = "q26_Q_seed%d_%s" % (Q.SEED, TAG)
resQ = go(lambda: Q.run_quorum(llm, Q.SEED, Q.STEPS, store.path(nameQ + ".jsonl"), meta=META, max_tokens=Q.Q_TOKEN_CAP, progress=persisting(nameQ, prog)))
if resQ:
    F.print_stats("ARM Q (witnesses)", resQ["rows"]); F.print_kinds(resQ["rows"]); Q.print_q(resQ["rows"]); store.persist(nameQ + ".jsonl")
    print("spent so far on this model: $%.2f" % llm.spent_usd()); files.download(save(resQ, nameQ))

### Cell 9: the READING. Replays every saved reply (a check that nothing was changed), prints the 16 lines with yes / NO and the headline, and saves the report. Send me the output and the downloaded files

In [ ]:
ok = {}
try: F.replay(resS["rows"], Q.SEED, "A"); ok["S"] = True
except Exception as e: print("replay S:", e); ok["S"] = False
try: Q.replay_q(resQ["rows"], Q.SEED); ok["Q"] = True
except Exception as e: print("replay Q:", e); ok["Q"] = False
checks = Q.verdict(resS, resQ, ok)
F.print_stats("ARM S (one witness)", resS["rows"]); F.print_stats("ARM Q (witnesses)", resQ["rows"]); Q.print_q(resQ["rows"])
Q.print_verdict(checks)
print("usage:", llm.usage(), "| spent: $%.2f" % llm.spent_usd())
files.download(save(dict(S=resS, Q=resQ, checks=checks, replays=ok, headline=Q.headline(checks)), "gemini_quorum_all_%s" % TAG))

### Done
Files downloaded: the arm S and arm Q files and `gemini_quorum_all_*.json`. Upload `gemini_quorum_all_minimal.json` to the chat. Nothing here says anything about any company's own service: it measures this model on this job.